In [0]:
file_path = "/Volumes/workspace/default/gharchive_raw/2026-09-01-0.json.gz"

df = spark.read.json(file_path)

print("Rows:", df.count())
df.printSchema()

In [0]:
from pyspark.sql import functions as F

# 1. Event type distribution
print("=== EVENT TYPES ===")
(
    df.groupBy("type")
      .count()
      .orderBy(F.desc("count"))
      .show(50, truncate=False)
)

# 2. Timestamp range
print("=== TIMESTAMP RANGE ===")
df.select(
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at")
).show(truncate=False)

# 3. Core-field nulls
print("=== CORE FIELD NULLS ===")
df.select(
    F.sum(F.col("id").isNull().cast("int")).alias("null_event_id"),
    F.sum(F.col("type").isNull().cast("int")).alias("null_type"),
    F.sum(F.col("created_at").isNull().cast("int")).alias("null_created_at"),
    F.sum(F.col("actor.id").isNull().cast("int")).alias("null_actor_id"),
    F.sum(F.col("repo.id").isNull().cast("int")).alias("null_repo_id")
).show()

# 4. Duplicate event IDs
print("=== DUPLICATE EVENT IDs ===")
duplicate_ids = (
    df.filter(F.col("id").isNotNull())
      .groupBy("id")
      .count()
      .filter(F.col("count") > 1)
)

print("Duplicate IDs:", duplicate_ids.count())

duplicate_ids.orderBy(F.desc("count")).show(20, truncate=False)

# 5. A small human-readable sample
print("=== SAMPLE EVENTS ===")
df.select(
    "id",
    "type",
    "created_at",
    "actor.id",
    "actor.login",
    "repo.id",
    "repo.name"
).show(10, truncate=False)

In [0]:
from pyspark.sql import functions as F

source_path = "/Volumes/workspace/default/gharchive_raw/2026-09-01-*.json.gz"

df_6h = (
    spark.read.json(source_path)
         .withColumn("source_file", F.col("_metadata.file_path"))
)

print("Total rows:", df_6h.count())

print("=== TIME RANGE ===")
df_6h.select(
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at")
).show(truncate=False)

print("=== EVENT TYPES ===")
(
    df_6h.groupBy("type")
         .count()
         .orderBy(F.desc("count"))
         .show(50, truncate=False)
)

print("=== ROWS PER SOURCE FILE ===")
(
    df_6h.groupBy("source_file")
         .count()
         .orderBy("source_file")
         .show(20, truncate=False)
)

print("=== DUPLICATE EVENT IDS ACROSS 6 HOURS ===")
duplicate_ids_6h = (
    df_6h.groupBy("id")
         .count()
         .filter(F.col("count") > 1)
)

print("Duplicate IDs:", duplicate_ids_6h.count())

In [0]:
from pyspark.sql import functions as F

print("=== PUSH EVENT ===")
(
    df_6h
    .filter(F.col("type") == "PushEvent")
    .select(
        "id",
        "created_at",
        F.col("actor.id").alias("actor_id"),
        F.col("repo.id").alias("repo_id"),
        F.col("repo.name").alias("repo_name"),
        F.col("payload.ref").alias("ref"),
        F.col("payload.before").alias("before"),
        F.col("payload.head").alias("head"),
        F.col("payload.push_id").alias("push_id")
    )
    .show(5, truncate=False)
)

print("=== PULL REQUEST EVENT ===")
(
    df_6h
    .filter(F.col("type") == "PullRequestEvent")
    .select(
        "id",
        "created_at",
        F.col("actor.id").alias("actor_id"),
        F.col("repo.id").alias("repo_id"),
        F.col("payload.action").alias("action"),
        F.col("payload.number").alias("pr_number"),
        F.col("payload.pull_request.id").alias("pr_id")
    )
    .show(5, truncate=False)
)

print("=== ISSUES EVENT ===")
(
    df_6h
    .filter(F.col("type") == "IssuesEvent")
    .select(
        "id",
        "created_at",
        F.col("actor.id").alias("actor_id"),
        F.col("repo.id").alias("repo_id"),
        F.col("payload.action").alias("action"),
        F.col("payload.issue.id").alias("issue_id"),
        F.col("payload.issue.number").alias("issue_number"),
        F.col("payload.issue.state").alias("state")
    )
    .show(5, truncate=False)
)

print("=== WATCH / FORK / RELEASE ===")
(
    df_6h
    .filter(F.col("type").isin("WatchEvent", "ForkEvent", "ReleaseEvent"))
    .select(
        "type",
        "id",
        "created_at",
        F.col("actor.id").alias("actor_id"),
        F.col("repo.id").alias("repo_id"),
        F.col("payload.action").alias("action"),
        F.col("payload.forkee.id").alias("forked_repo_id"),
        F.col("payload.release.id").alias("release_id")
    )
    .show(20, truncate=False)
)